# L4 40 — run the matched-layout Qwen 3B pilot

Runs the gated repaired pilot in IPD: 8 fresh matched seeds, 12 rounds, and all six conditions (`none`, readable `text`, `trained`, `random`, `zero`, `shuffled`).

Readable text-token embeddings and every latent payload occupy the exact same position after an identical receiver prompt. The run is exploratory, checkpointed to Drive, and safe to resume.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True))
for name in ('HF_TOKEN', 'L4_RECEIVER_URL', 'L4_RECEIVER_TOKEN'):
    try:
        value = userdata.get(name)
        if value: os.environ[name] = value
    except Exception:
        print(f'{name}: not configured (receiver secrets are optional)')

In [ ]:
REPO = 'https://github.com/harrywinner2/rival-arena-capstone.git'
REVISION = 'fd338680220c01d89e1452d77ae3a72fcf902f62'
WORK = pathlib.Path('/content/rival-arena-capstone')
if not WORK.exists(): subprocess.run(['git', 'clone', REPO, str(WORK)], check=True)
subprocess.run(['git', '-C', str(WORK), 'fetch', '--all'], check=True)
subprocess.run(['git', '-C', str(WORK), 'checkout', REVISION], check=True)
subprocess.run(['pip', 'install', '-q', '-r', str(WORK/'followup-representational/requirements.txt')], check=True)

In [ ]:
JOB_ID = 'matched-qwen3b-t4-001'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
PROFILE = 'matched_pilot'
SEED_OFFSET = 800
JOB_DIR = pathlib.Path('/content/drive/MyDrive/rival-arena-l4') / JOB_ID
for required in (
    JOB_DIR/'faithful_link.pt',
    JOB_DIR/'validation_report.json',
    JOB_DIR/'arena_context_fidelity_matched_v2/arena_context_fidelity_report.json',
):
    assert required.exists(), f'Missing required Drive artifact: {required}'
print('Job:', JOB_ID, '| profile:', PROFILE, '| seeds:', SEED_OFFSET, 'through', SEED_OFFSET + 7)

In [ ]:
command = [
    'python', 'scripts/run_arena.py',
    '--model', MODEL,
    '--job-dir', JOB_DIR,
    '--job-id', JOB_ID,
    '--profile', PROFILE,
    '--seed-offset', str(SEED_OFFSET),
]
print(' '.join(map(str, command)))
subprocess.run(list(map(str, command)), cwd=WORK/'followup-representational', check=True, env=os.environ)

## Completion

Expected output: `MyDrive/rival-arena-l4/matched-qwen3b-t4-001/arena_matched_pilot_v4/`. It contains 48 checkpointed matches. Do not launch a confirmatory run until this pilot is reviewed.